# Structured streaming - Window explorations

- **Tumbling window** (`window(col, "5 minutes")`): fixed, non-overlapping intervals, each event belongs to exactly one window.
- **Sliding window** (`window(col, "10 minutes", "5 minutes")`): 10-minute windows starting every 5 minutes, so they overlap and each event belongs to two windows.

Results are written to an in-memory table (`memory` sink) so they can be queried with SQL directly in the notebook.

In [1]:
# Imports cell

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, from_json
from pyspark.sql.types import StructType, StructField, StringType, LongType, IntegerType, BooleanType

from pyspark.sql.functions import window, col, count

# Convert timestamp (Unix epoch in seconds) to timestamp type
from pyspark.sql.functions import from_unixtime




In [2]:
# initialize the spark session
spark = SparkSession.builder \
    .appName("KafkaStreamingPrototype") \
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.13:4.2.0") \
    .getOrCreate()


In [3]:
# Define the initial raw streaming dataframe
raw_df = spark \
    .readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "kafka:19092") \
    .option("subscribe", "wikistreams") \
    .option("startingOffsets", "earliest") \
    .load()


In [4]:
# Define a simplified schema - just the key fields for analysis
simple_wikimedia_schema = StructType([
    StructField("title", StringType(), True),
    StructField("user", StringType(), True),
    StructField("timestamp", LongType(), True),
    StructField("type", StringType(), True),
    StructField("wiki", StringType(), True),
    StructField("bot", BooleanType(), True),
    StructField("minor", BooleanType(), True),
    StructField("comment", StringType(), True),
    StructField("length", StructType([
        StructField("old", IntegerType(), True),
        StructField("new", IntegerType(), True)
    ]), True)
])

# Apply the schema to organize the raw input
parsed_df = raw_df.select(
    from_json(col("value").cast("string"), simple_wikimedia_schema).alias("data")
).select("data.*")

In [5]:
# Enrich the parsed data: real timestamp type + size of each edit in bytes
from pyspark.sql.functions import abs, avg, sum, max, min, round, coalesce, lit, when, approx_count_distinct

enriched_df = parsed_df     .withColumn("event_time", col("timestamp").cast("timestamp"))     .withColumn("size_diff", coalesce(col("length.new"), lit(0)) - coalesce(col("length.old"), lit(0)))

In [6]:
# Exploration 1 - TUMBLING window (5 min, no overlap): volume and size of edits, bots vs. humans
tumbling_df = enriched_df     .groupBy(window(col("event_time"), "5 minutes"), col("bot"))     .agg(
        count("*").alias("edit_count"),
        sum(abs(col("size_diff"))).alias("total_bytes_changed"),
        round(avg(col("size_diff")), 1).alias("avg_size_diff"),
        max(col("size_diff")).alias("biggest_addition"),
        min(col("size_diff")).alias("biggest_removal")
    )     .select(
        col("window.start").alias("window_start"),
        col("window.end").alias("window_end"),
        "bot", "edit_count", "total_bytes_changed", "avg_size_diff", "biggest_addition", "biggest_removal"
    )

tumbling_query = tumbling_df     .writeStream     .format("memory")     .queryName("tumbling_edits")     .outputMode("complete")     .start()

In [7]:
# Exploration 2 - SLIDING window (10 min, sliding every 5 min, overlapping): activity and diversity
sliding_df = enriched_df     .groupBy(window(col("event_time"), "10 minutes", "5 minutes"))     .agg(
        count("*").alias("edit_count"),
        approx_count_distinct(col("user")).alias("distinct_users"),
        approx_count_distinct(col("title")).alias("distinct_pages"),
        sum(when(col("minor"), 1).otherwise(0)).alias("minor_edits")
    )     .select(
        col("window.start").alias("window_start"),
        col("window.end").alias("window_end"),
        "edit_count", "distinct_users", "distinct_pages", "minor_edits"
    )

sliding_query = sliding_df     .writeStream     .format("memory")     .queryName("sliding_activity")     .outputMode("complete")     .start()

In [8]:
# Wait for the first micro-batch of each query to be processed
tumbling_query.processAllAvailable()
sliding_query.processAllAvailable()

In [9]:
# Display the current results (re-run this cell to see them update as new events arrive)
print("Tumbling window (5 min):")
spark.sql("SELECT * FROM tumbling_edits ORDER BY window_start, bot").show(50, truncate=False)

print("Sliding window (10 min, every 5 min):")
spark.sql("SELECT * FROM sliding_activity ORDER BY window_start").show(50, truncate=False)

Tumbling window (5 min):
+-------------------+-------------------+-----+----------+-------------------+-------------+----------------+---------------+
|window_start       |window_end         |bot  |edit_count|total_bytes_changed|avg_size_diff|biggest_addition|biggest_removal|
+-------------------+-------------------+-----+----------+-------------------+-------------+----------------+---------------+
|2026-10-01 05:40:00|2026-10-01 05:45:00|false|8         |814                |7.0          |400             |-271           |
|2026-10-01 05:40:00|2026-10-01 05:45:00|true |15        |5054               |192.5        |1042            |-1040          |
|2026-10-01 05:45:00|2026-10-01 05:50:00|false|34        |13295              |354.6        |4486            |-185           |
|2026-10-01 05:45:00|2026-10-01 05:50:00|true |79        |91708              |853.6        |45102           |-2368          |
|2026-10-01 05:50:00|2026-10-01 05:55:00|false|45        |5400               |99.2         |1

In [10]:
# Stop all streaming queries when done
for q in spark.streams.active:
    q.stop()